# DipDECK reconstruction-loss check

The standalone's DipDECK fits pass `torch.nn.MSELoss(reduction="mean")`, which is the paper's
Equation 1 divided by the input width. The library default at ClustPy `82548dc` is Equation 1
itself (`clustpy.deep._utils.mean_squared_error`). This notebook asks whether that choice, rather
than the input width, makes raw ESM-2 collapse.

It refits the paper-default reference (initial clusters 35, threshold 0.90, 100 plus nominal 50
epochs, the 1,577 GraphPart-retained records) on raw ProtT5 1,024 and raw ESM-2 3B 2,560, seeds 0
to 9, with **only the loss changed**: 20 fits. The estimator comes from the standalone's own
`_create_dipdeck`, so every other argument matches the stored references. Two control fits keep
the original loss at seed 0, one per embedding, to show whether this route reproduces the stored
seed-0 references. About 6 minutes of fitting on a T4 (MEASURED: a stored reference fit took
15 to 17 seconds).

**Reading fixed before any fit.** Under the library loss, compare raw ESM-2
with raw ProtT5:

- The loss explains the collapse if ESM-2's median cluster count is at least the lowest ProtT5
  count and its median largest-cluster share is at most the highest ProtT5 share.
- Width or the model still matters if ESM-2's median count is below the lowest ProtT5 count and
  its median share is above the highest ProtT5 share.
- Otherwise the result is mixed.

With the original loss the stored references read "still matters" (ESM-2 median 8.5 clusters and
63.7% largest share, against ProtT5's 12 to 19 and 23.0% to 54.9%).

**To run:** upload this project folder to Google Drive as `MyDrive/explaining-plm-latent-spaces` (or set `PROJECT_DIR`). Runtime > Change runtime type > T4 GPU. Use a fresh runtime, then Runtime > Run all. If the install cell says torch
is already imported, use Runtime > Restart session and Run all again. The last cell prints the ZIP
name and its SHA256. A rerun deletes this notebook's earlier output first, so old and new fits never
mix.


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/explaining-plm-latent-spaces"
CLUSTPY_COMMIT = "82548dc35a01d221474fbb44908b3621d4b97616"
EXPECTED_UPSTREAM_DIPDECK_SHA256 = "c567d24c1722087a75c3eb56f17fe91152b6f818148083d908086b9fc78d438a"
EXPECTED_DIPDECK_SHA256 = "cfbf629ab341bdbb5967af232c24cfc3f2e5fb09cd082fb28b317c45738b7420"
DATA_SHA256 = {
    "data/phosphatase/phosphatase/esm2_t36_3B_UR50D_full_length.h5":
        "00c27d5730ba59db9ca4178118a2ba5fc4e4e4ecc41266afdc9e000951e8571c",
    "data/phosphatase/phosphatase/tmp/prot_t5.h5":
        "425e005d783d70c5ac9043ad2c7d794ce603a8ddbecffa405d569aeddb78c7c1",
}
EMBEDDINGS = ("prott5", "esm2_3b")
SEEDS = tuple(range(10))
RUN_CONTROL = True
DEVICE = "cuda"
DRIVE_ROOT = PROJECT_DIR
OUTPUT_FOLDER = "loss_check"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import os
import shutil
import subprocess

repo = Path("/content/project").resolve()
clustpy_source = Path("/content/ClustPy").resolve()
drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)

for target in (repo, clustpy_source):
    if target.parent != Path("/content"):
        raise RuntimeError(f"Refusing to replace unexpected path: {target}")
    if target.exists():
        shutil.rmtree(target)

project_dir = Path(PROJECT_DIR)
if not (project_dir / "src/algo/dipdeck/dipdeck_standalone.py").is_file():
    raise FileNotFoundError(f"Upload this project folder to Google Drive at {project_dir}, then rerun.")
for part in ("src", "data", "results/graphpart", "results/ec_audit"):
    shutil.copytree(project_dir / part, repo / part)
subprocess.run(["git", "clone", "https://github.com/collinleiber/ClustPy.git", str(clustpy_source)], check=True)
subprocess.run(["git", "-C", str(clustpy_source), "checkout", CLUSTPY_COMMIT], check=True)

dipdeck_source = clustpy_source / "clustpy/deep/dipdeck.py"
source_text = dipdeck_source.read_text(encoding="utf-8")
upstream_sha256 = hashlib.sha256(source_text.encode()).hexdigest()
if upstream_sha256 != EXPECTED_UPSTREAM_DIPDECK_SHA256:
    raise RuntimeError(
        f"Unexpected upstream DipDECK source: expected {EXPECTED_UPSTREAM_DIPDECK_SHA256}, "
        f"got {upstream_sha256}"
    )
for old, new in (
    ("self.labels.to(device)", "self.labels = self.labels.to(device)"),
    ("self.cluster_centers.to(device)", "self.cluster_centers = self.cluster_centers.to(device)"),
    ("self.dip_matrix.to(device)", "self.dip_matrix = self.dip_matrix.to(device)"),
):
    if source_text.count(old) != 1:
        raise RuntimeError(f"CUDA patch target is absent or ambiguous: {old}")
    source_text = source_text.replace(old, new)
dipdeck_source.write_text(source_text, encoding="utf-8")
patched_sha256 = hashlib.sha256(dipdeck_source.read_bytes()).hexdigest()
if patched_sha256 != EXPECTED_DIPDECK_SHA256:
    raise RuntimeError(
        f"Patched DipDECK source mismatch: expected {EXPECTED_DIPDECK_SHA256}, got {patched_sha256}"
    )

repo_commit = "project-folder"

for relative, expected in DATA_SHA256.items():
    path = repo / relative
    if not path.is_file():
        raise FileNotFoundError(f"Project copy is missing {relative}")
    observed = hashlib.sha256(path.read_bytes()).hexdigest()
    if observed != expected:
        raise RuntimeError(f"{relative}: expected SHA256 {expected}, got {observed}")
if (repo / "data/phosphatase/phosphatase/data_standard.parquetbundle").is_file() is False:
    raise FileNotFoundError("Project copy is missing the phosphatase annotation bundle")

os.chdir(repo)
print(f"Repository: {repo}")
print(f"Project copied from: {PROJECT_DIR}")
print(f"Data checked: {list(DATA_SHA256)}")
print(f"ClustPy revision: {CLUSTPY_COMMIT}")

In [ ]:
COLAB_PINS = {
    "Cython": "3.3.0",
    "numpy": "2.1.3",
    "scipy": "1.18.1",
    "scikit-learn": "1.6.1",
    "pandas": "2.2.3",
    "h5py": "3.16.0",
    "pyarrow": "25.0.1",
    "tqdm": "4.70.0",
    "pytest": "9.1.1",
    "psutil": "7.2.2",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
}

import os
import subprocess
import sys

os.chdir("/content")
install_names = [
    "Cython", "numpy", "scipy", "scikit-learn", "pandas",
    "h5py", "pyarrow", "tqdm", "pytest", "psutil",
]
requirements = [f"{name}=={COLAB_PINS[name]}" for name in install_names]
install_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", *requirements],
    check=True, capture_output=True, text=True,
)
print(install_result.stdout)
if install_result.stderr:
    print(install_result.stderr, file=sys.stderr)

if "torch" in sys.modules or "torchvision" in sys.modules:
    raise RuntimeError("torch is already imported. Use Runtime > Restart session, then run all cells.")
torch_result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        f"torch=={COLAB_PINS['torch']}", f"torchvision=={COLAB_PINS['torchvision']}",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ],
    check=True, capture_output=True, text=True,
)
print(torch_result.stdout[-2000:])
if torch_result.stderr:
    print(torch_result.stderr[-2000:], file=sys.stderr)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "--force-reinstall", "--no-cache-dir", "--editable", str(clustpy_source),
] , check=True)

pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True,
)
print(pip_check.stdout)
if pip_check.stderr:
    print(pip_check.stderr, file=sys.stderr)
if pip_check.returncode != 0:
    print("pip check reported the Colab base-environment warning above.", file=sys.stderr)

In [ ]:
import hashlib
import importlib
import importlib.metadata as metadata
import json
import os
import platform
import sys
from pathlib import Path

for module_name in (
    "Cython", "h5py", "pandas", "psutil", "pyarrow", "pytest", "scipy", "sklearn", "tqdm",
):
    importlib.import_module(module_name)

import numpy as np
import torch
import torchvision
from threadpoolctl import threadpool_info

if not (clustpy_source / "clustpy/__init__.py").is_file():
    raise FileNotFoundError(f"ClustPy package is missing from the verified checkout: {clustpy_source}")
sys.path.insert(0, str(clustpy_source))
import clustpy
from clustpy.deep import DipDECK
if Path(clustpy.__file__).resolve().parent != (clustpy_source / "clustpy").resolve():
    raise RuntimeError(f"Imported ClustPy is not the verified checkout: {clustpy.__file__}")

observed_versions = {name: metadata.version(name) for name in COLAB_PINS}
version_mismatches = {
    name: {"expected": expected, "observed": observed_versions[name]}
    for name, expected in COLAB_PINS.items()
    if observed_versions[name] != expected
}
if version_mismatches:
    raise RuntimeError(f"Colab dependency mismatch: {version_mismatches}")

dipdeck_source = clustpy_source / "clustpy/deep/dipdeck.py"
dipdeck_sha256 = hashlib.sha256(dipdeck_source.read_bytes()).hexdigest()
if dipdeck_sha256 != EXPECTED_DIPDECK_SHA256:
    raise RuntimeError(
        f"DipDECK source mismatch: expected {EXPECTED_DIPDECK_SHA256}, got {dipdeck_sha256}"
    )

environment = {
    "python": platform.python_version(),
    "project_commit": repo_commit,
    "clustpy_version": metadata.version("clustpy"),
    "clustpy_path": clustpy.__file__,
    "clustpy_commit": CLUSTPY_COMMIT,
    "dipdeck_sha256": dipdeck_sha256,
    "package_versions": observed_versions,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_num_threads": torch.get_num_threads(),
    "torch_num_interop_threads": torch.get_num_interop_threads(),
    "thread_environment": {
        name: os.environ.get(name)
        for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    },
    "threadpool_info": threadpool_info(),
}
print(json.dumps(environment, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. In Colab choose Runtime > Change runtime type > GPU, then run all again.")

In [ ]:
import contextlib
import inspect
import io
import json
import shutil
import sys
import time

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(repo / "src"))
sys.path.insert(0, str(repo / "src/algo/dipdeck"))
import dipdeck_standalone as ds
from clustpy.deep import DipDECK
from clustpy.deep._utils import mean_squared_error as library_loss

if inspect.signature(DipDECK.__init__).parameters["ssl_loss_fn"].default is not library_loss:
    raise RuntimeError("ClustPy's default ssl_loss_fn is not clustpy.deep._utils.mean_squared_error")

LOSSES = {
    "original_mse_mean": None,
    "library_default": library_loss,
}
FITS = (
    [(embedding, "original_mse_mean", 0) for embedding in EMBEDDINGS if RUN_CONTROL]
    + [(embedding, "library_default", seed) for embedding in EMBEDDINGS for seed in SEEDS]
)

output_root = (drive_root / "results" / OUTPUT_FOLDER).resolve()
if output_root.exists():
    shutil.rmtree(output_root)
(output_root / "assignments").mkdir(parents=True)
(output_root / "logs").mkdir()


def reference_input(embedding):
    dataset = ds._load_embedding_dataset(embedding)
    reference = ds._scientific_reference_dataset(dataset)
    X, ids = ds._canonical_input(reference.X, reference.ids)
    if len(ids) != ds.SCIENTIFIC_RETAINED_N or reference.subset_hash != "20c12a04f45d":
        raise RuntimeError(f"{embedding}: records do not match the stored references")
    return X, ids, reference.subset_hash


inputs = {embedding: reference_input(embedding) for embedding in EMBEDDINGS}
rows = []
for embedding, loss_name, seed in FITS:
    X, ids, subset_hash = inputs[embedding]
    config = ds._scientific_reference_configs((seed,))[0]
    estimator, runtime = ds._create_dipdeck(config, DEVICE, clustpy_source)
    standalone_loss = estimator.ssl_loss_fn
    if not (isinstance(standalone_loss, torch.nn.MSELoss) and standalone_loss.reduction == "mean"):
        raise RuntimeError("the standalone no longer passes MSELoss(reduction='mean'); this check is stale")
    if LOSSES[loss_name] is not None:
        estimator.ssl_loss_fn = LOSSES[loss_name]
    row = f"dipdeck_loss_check_{ds.EMBEDDINGS[embedding].raw_space}_{loss_name}_k035_t0p9_s{seed:03d}"
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream), contextlib.redirect_stderr(stream):
        with ds._count_dipdeck_clustering_epochs(estimator) as updates:
            started = time.perf_counter()
            estimator.fit(X)
            elapsed = time.perf_counter() - started
    labels = np.asarray(estimator.labels_, dtype=np.int32)
    sizes = np.bincount(labels)
    np.savez_compressed(output_root / "assignments" / f"{row}.npz", protein_id=ids.astype(str), label=labels)
    (output_root / "logs" / f"{row}.log").write_text(stream.getvalue(), encoding="utf-8")
    rows.append({
        "row": row, "embedding": embedding, "input_space": ds.EMBEDDINGS[embedding].raw_space,
        "loss": loss_name,
        "ssl_loss_fn": getattr(estimator.ssl_loss_fn, "__name__", type(estimator.ssl_loss_fn).__name__),
        "seed": seed, "n": len(ids), "subset_hash": subset_hash,
        "n_clusters_init": config.n_clusters_init, "dip_merge_threshold": config.dip_merge_threshold,
        "pretrain_epochs": config.pretrain_epochs, "clustering_epochs": config.clustering_epochs,
        "n_clusters": int(estimator.n_clusters_), "n_label_values": int(np.count_nonzero(sizes)),
        "largest_cluster_size": int(sizes.max()), "largest_cluster_share": float(sizes.max() / len(ids)),
        "merge_count": sum(bool(ds.MERGE_RECORD.match(line)) for line in stream.getvalue().splitlines()),
        "clustering_epoch_updates": updates[0] if updates is not None else "",
        "elapsed_seconds": elapsed, "project_commit": repo_commit, "clustpy_commit": CLUSTPY_COMMIT,
        "dipdeck_sha256": EXPECTED_DIPDECK_SHA256, "device": DEVICE,
        "device_name": runtime["device_name"], "deterministic_algorithms": runtime["deterministic_algorithms"],
        "torch_version": torch.__version__, "numpy_version": np.__version__,
        "sklearn_version": ds._package_version("scikit-learn"),
    })
    print(f"{row}: {rows[-1]['n_clusters']} clusters, largest {rows[-1]['largest_cluster_share']:.3f}, "
          f"{elapsed:.1f} s")

runs = pd.DataFrame(rows)
runs.to_csv(output_root / "loss_check_runs.csv", index=False)
(output_root / "manifest.json").write_text(json.dumps({
    "fits": len(runs), "embeddings": list(EMBEDDINGS), "seeds": list(SEEDS),
    "control": RUN_CONTROL, "project_commit": repo_commit, "clustpy_commit": CLUSTPY_COMMIT,
    "dipdeck_sha256": EXPECTED_DIPDECK_SHA256, "rows": runs["row"].tolist(),
}, indent=2) + "\n", encoding="utf-8")
print(f"{len(runs)} fits written to {output_root}")


In [ ]:
import hashlib
import os
import zipfile

summary = runs.groupby(["embedding", "loss"]).agg(
    fits=("row", "size"),
    clusters_median=("n_clusters", "median"), clusters_min=("n_clusters", "min"),
    clusters_max=("n_clusters", "max"),
    largest_share_median=("largest_cluster_share", "median"),
    largest_share_min=("largest_cluster_share", "min"),
    largest_share_max=("largest_cluster_share", "max"),
)
print(summary.round(3).to_string())

STORED = {"prott5": "scientific_reference", "esm2_3b": "scientific_reference_esm2_3b_raw2560"}
for embedding in EMBEDDINGS:
    control = runs[(runs.embedding == embedding) & (runs.loss == "original_mse_mean")]
    if control.empty:
        continue
    stored = pd.read_csv(repo / f"results/dipdeck/{STORED[embedding]}/seed_000/metrics/dipdeck_standalone_runs.csv")
    print(f"control {embedding}: {int(control.n_clusters.iloc[0])} clusters here, "
          f"{int(stored.n_clusters.iloc[0])} in the stored seed-0 reference")

archive_base = drive_root / f"dipdeck_{OUTPUT_FOLDER}_outputs"
partial = Path(f"{archive_base}_partial.zip")
partial.unlink(missing_ok=True)
shutil.make_archive(str(partial.with_suffix("")), "zip", root_dir=output_root.parent, base_dir=OUTPUT_FOLDER)
expected_files = sum(1 for path in output_root.rglob("*") if path.is_file())
with zipfile.ZipFile(partial) as packed:
    if packed.testzip() is not None:
        raise RuntimeError(f"Corrupt member in {partial}")
    packed_files = sum(1 for name in packed.namelist() if not name.endswith("/"))
if packed_files != expected_files:
    raise RuntimeError(f"{partial} holds {packed_files} files, the folder has {expected_files}")
archive = archive_base.with_suffix(".zip")
os.replace(partial, archive)
print(f"Output archive: {archive}")
print(f"Output SHA256: {hashlib.sha256(archive.read_bytes()).hexdigest()}")
